In [15]:
import pandas as pd
import numpy as np
from pathlib import Path

# Diretórios base do projeto
project_root = Path("..").resolve()
processed_path = project_root / "data" / "processed"
output_path = project_root / "data" / "output"
output_path.mkdir(parents=True, exist_ok=True)

# Carregando dataframes processados
# Cada arquivo já foi tratado no pipeline anterior; aqui apenas consolidamos a análise.
df_orders = pd.read_csv(processed_path / "orders_processed.csv", encoding="UTF-8")
df_items = pd.read_csv(processed_path / "order_items_processed.csv", encoding="UTF-8")
df_payments = pd.read_csv(processed_path / "order_payments_processed.csv", encoding="UTF-8")
df_reviews = pd.read_csv(processed_path / "order_reviews_processed.csv", encoding="UTF-8")
df_products = pd.read_csv(processed_path / "products_processed.csv", encoding="UTF-8")
df_sellers = pd.read_csv(processed_path / "sellers_processed.csv", encoding="UTF-8")
df_customers = pd.read_csv(processed_path / "customers_processed.csv", encoding="UTF-8")

In [16]:
# Bloco 1: conversão de colunas temporais para facilitar dashboards e agregações mensais
# A padronização do tipo datetime é essencial para análise temporal e cálculo de SLA.
for df, datetime_cols in [
    (df_orders, ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"]),
    (df_items, ["shipping_limit_date"]),
    (df_reviews, ["review_creation_date", "review_answer_timestamp"]),
]:
    for col in datetime_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

# Extração de componentes de data para uso em Tableau e filtros
# Isso permite responder perguntas como: receita por mês, pedidos por dia da semana e atraso por região.
df_orders = df_orders.assign(
    purchase_year=df_orders["order_purchase_timestamp"].dt.year,
    purchase_month=df_orders["order_purchase_timestamp"].dt.month,
    purchase_month_name=df_orders["order_purchase_timestamp"].dt.month_name(locale="pt_BR"),
    purchase_day=df_orders["order_purchase_timestamp"].dt.day,
    purchase_dayofweek=df_orders["order_purchase_timestamp"].dt.dayofweek,
    purchase_weekday_name=df_orders["order_purchase_timestamp"].dt.day_name(),
    purchase_hour=df_orders["order_purchase_timestamp"].dt.hour,
    approval_year=df_orders["order_approved_at"].dt.year,
    approval_month=df_orders["order_approved_at"].dt.month,
    approval_day=df_orders["order_approved_at"].dt.day,
    carrier_year=df_orders["order_delivered_carrier_date"].dt.year,
    carrier_month=df_orders["order_delivered_carrier_date"].dt.month,
    carrier_day=df_orders["order_delivered_carrier_date"].dt.day,
    delivered_year=df_orders["order_delivered_customer_date"].dt.year,
    delivered_month=df_orders["order_delivered_customer_date"].dt.month,
    delivered_day=df_orders["order_delivered_customer_date"].dt.day,
    estimated_year=df_orders["order_estimated_delivery_date"].dt.year,
    estimated_month=df_orders["order_estimated_delivery_date"].dt.month,
    estimated_day=df_orders["order_estimated_delivery_date"].dt.day,
)

# Corrigindo possível incompatibilidade de locale em alguns ambientes.
# Se necessário, a coluna de nome do mês pode ser usada sem depender do locale.
df_orders["purchase_month_name"] = df_orders["order_purchase_timestamp"].dt.strftime("%b")

# Cálculo de métricas de logística e satisfação
# Estas colunas ajudam na análise de atraso e relação com avaliações.
df_orders["delivery_days"] = (
    (df_orders["order_delivered_customer_date"] - df_orders["order_purchase_timestamp"]).dt.total_seconds() / 86400
)
df_orders["estimated_delivery_days"] = (
    (df_orders["order_estimated_delivery_date"] - df_orders["order_purchase_timestamp"]).dt.total_seconds() / 86400
)
df_orders["delay_days"] = (
    (df_orders["order_delivered_customer_date"] - df_orders["order_estimated_delivery_date"]).dt.total_seconds() / 86400
)

df_orders["is_delayed"] = (df_orders["delay_days"] > 0).astype(int)

df_orders["delay_flag"] = np.where(df_orders["delay_days"] > 0, "Atrasado", "NoPrazo")

In [17]:
# Bloco 2: agregação de pagamentos por pedido para facilitar a análise financeira
# A chave de junção é order_id; cada pedido pode ter um ou mais pagamentos.
df_payments_agg = (
    df_payments.groupby("order_id", as_index=False)
    .agg(
        payment_total=("payment_value", "sum"),
        payment_methods_count=("payment_type", "nunique"),
        payment_types=("payment_type", lambda s: ", ".join(s.dropna().astype(str).unique()))
    )
)
display(df_payments_agg)

,order_id,payment_total,payment_methods_count,payment_types
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,credit_card
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,credit_card
2,000229ec398224ef6ca0657da4fc703e,216.87,1,credit_card
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,credit_card
...,...,...,...,...
99435,fffc94f6ce00a00581880bf54a75a037,343.40,1,boleto
99436,fffcd46ef2263f404302a634eb57f7eb,386.53,1,boleto
99437,fffce4705a9662cd70adb13d4a31832d,116.85,1,credit_card
99438,fffe18544ffabc95dfada21779c9644f,64.71,1,credit_card


In [18]:
# Bloco 3: agregação de itens por pedido para compor a visão de Receita e volume por pedido
# Isso permite calcular ticket médio, itens por pedido e valor bruto/logístico.
df_items_agg = (
    df_items.groupby("order_id", as_index=False)
    .agg(
        items_quantity=("order_item_id", "count"),
        total_price=("price", "sum"),
        total_freight=("freight_value", "sum")
    )
)
display(df_items_agg)

,order_id,items_quantity,total_price,total_freight
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14
...,...,...,...,...
98661,fffc94f6ce00a00581880bf54a75a037,1,299.99,43.41
98662,fffcd46ef2263f404302a634eb57f7eb,1,350.00,36.53
98663,fffce4705a9662cd70adb13d4a31832d,1,99.90,16.95
98664,fffe18544ffabc95dfada21779c9644f,1,55.99,8.72


In [19]:
# Bloco 4: agregação de avaliações por pedido para responder relação entre qualidade e logística
# Como cada pedido pode ter eventualmente uma avaliação, usamos left join preservando pedidos sem revisão.
df_reviews_agg = (
    df_reviews.groupby("order_id", as_index=False)
    .agg(
        review_score_mean=("review_score", "mean"),
        review_score_max=("review_score", "max"),
        review_score_min=("review_score", "min"),
        review_count=("review_id", "count")
    )
)
display(df_reviews_agg)

,order_id,review_score_mean,review_score_max,review_score_min,review_count
0,00010242fe8c5a6d1ba2dd792cb16214,5.0,5,5,1
1,00018f77f2f0320c557190d7a144bdd3,4.0,4,4,1
2,000229ec398224ef6ca0657da4fc703e,5.0,5,5,1
3,00024acbcdf0a6daa1e931b038114c75,4.0,4,4,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0,5,5,1
...,...,...,...,...,...
98668,fffc94f6ce00a00581880bf54a75a037,5.0,5,5,1
98669,fffcd46ef2263f404302a634eb57f7eb,5.0,5,5,1
98670,fffce4705a9662cd70adb13d4a31832d,5.0,5,5,1
98671,fffe18544ffabc95dfada21779c9644f,5.0,5,5,1


In [20]:
# Bloco 5: joins principais do modelo analítico
# A abordagem de camada analítica mantém os joins organizados por responsabilidade.
# 5.1 Pedido + cliente
orders_customers = df_orders.merge(df_customers, on="customer_id", how="left", suffixes=("_order", "_customer"))
display(orders_customers)

# 5.2 Pedido + itens agregados
orders_items = orders_customers.merge(df_items_agg, on="order_id", how="left")
display(orders_items)

# 5.3 Pedido + pagamentos agregados
orders_items_payments = orders_items.merge(df_payments_agg, on="order_id", how="left")
display(orders_items_payments)

# 5.4 Pedido + avaliações agregados
orders_items_payments_reviews = orders_items_payments.merge(df_reviews_agg, on="order_id", how="left")
display(orders_items_payments_reviews)

# 5.5 Pedido + produto + vendedor
# Aqui usamos order_id como base para enriquecer com produto e vendedor de cada item.
# Como um pedido pode ter múltiplos itens, o join direto em df_items preserva a granularidade de item.
model_df = orders_items_payments_reviews.merge(df_items, on="order_id", how="left")
model_df = model_df.merge(df_products, on="product_id", how="left", suffixes=("_item", "_product"))
model_df = model_df.merge(df_sellers, on="seller_id", how="left", suffixes=("_item", "_seller"))

# Ajustes de tipos numéricos e preenchimento
model_df["payment_total"] = model_df["payment_total"].fillna(0)
model_df["total_price"] = model_df["total_price"].fillna(0)
model_df["total_freight"] = model_df["total_freight"].fillna(0)
model_df["review_score_mean"] = model_df["review_score_mean"].fillna(0)
model_df["review_count"] = model_df["review_count"].fillna(0)

# Como o schema processado não possui a coluna `quantity`, representamos a unidade do item por linha.
# Isso mantém a granularidade de item e permite calcular a receita por linha do pedido.
model_df["quantity"] = 1
model_df["item_revenue"] = model_df["price"] * model_df["quantity"]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,purchase_year,purchase_month,...,estimated_day,delivery_days,estimated_delivery_days,delay_days,is_delayed,delay_flag,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,Delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2017,10,...,18,8.436574,15.544063,-7.107488,0,NoPrazo,7c396fd4830fd04220f754e42b4e5bff,3149,Sao Paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,Delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,2018,7,...,13,13.782037,19.137766,-5.355729,0,NoPrazo,af07308b275d755c9edb36a90c618231,47813,Barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,Delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,2018,8,...,4,9.394213,26.639711,-17.245498,0,NoPrazo,3a653a41f6f9fc3d2a113cf8398680e8,75265,Vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,Delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,2017,11,...,15,13.208750,26.188819,-12.980069,0,NoPrazo,7c142cf63193a1473d2e66489a9ae977,59296,Sao Goncalo Do Amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,Delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2018,2,...,26,2.873877,12.112049,-9.238171,0,NoPrazo,72632f0f9dd73dfee390c9b22eb56dd6,9195,Santo Andre,SP
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,Delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28,2017,3,...,28,8.218009,18.587442,-10.369433,0,NoPrazo,6359f309b166b0196dbf7ad2ac62bb5a,12209,Sao Jose Dos Campos,SP
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,Delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02,2018,2,...,2,22.193727,23.459051,-1.265324,0,NoPrazo,da62f9e57a76d978d02ab5362c509660,11722,Praia Grande,SP
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,Delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27,2017,8,...,27,24.859421,30.384225,-5.524803,0,NoPrazo,737520a9aad80b3fbbdad19b66b37b30,45920,Nova Vicosa,BA
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,Delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15,2018,1,...,15,17.086424,37.105243,-20.018819,0,NoPrazo,5097a5312c8b157bb7be58ae360ef43c,28685,Japuiba,RJ


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,purchase_year,purchase_month,...,delay_days,is_delayed,delay_flag,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,items_quantity,total_price,total_freight
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,Delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2017,10,...,-7.107488,0,NoPrazo,7c396fd4830fd04220f754e42b4e5bff,3149,Sao Paulo,SP,1.0,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,Delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,2018,7,...,-5.355729,0,NoPrazo,af07308b275d755c9edb36a90c618231,47813,Barreiras,BA,1.0,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,Delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,2018,8,...,-17.245498,0,NoPrazo,3a653a41f6f9fc3d2a113cf8398680e8,75265,Vianopolis,GO,1.0,159.90,19.22
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,Delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,2017,11,...,-12.980069,0,NoPrazo,7c142cf63193a1473d2e66489a9ae977,59296,Sao Goncalo Do Amarante,RN,1.0,45.00,27.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,Delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2018,2,...,-9.238171,0,NoPrazo,72632f0f9dd73dfee390c9b22eb56dd6,9195,Santo Andre,SP,1.0,19.90,8.72
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,Delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28,2017,3,...,-10.369433,0,NoPrazo,6359f309b166b0196dbf7ad2ac62bb5a,12209,Sao Jose Dos Campos,SP,1.0,72.00,13.08
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,Delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02,2018,2,...,-1.265324,0,NoPrazo,da62f9e57a76d978d02ab5362c509660,11722,Praia Grande,SP,1.0,174.90,20.10
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,Delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27,2017,8,...,-5.524803,0,NoPrazo,737520a9aad80b3fbbdad19b66b37b30,45920,Nova Vicosa,BA,1.0,205.99,65.02
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,Delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15,2018,1,...,-20.018819,0,NoPrazo,5097a5312c8b157bb7be58ae360ef43c,28685,Japuiba,RJ,2.0,359.98,81.18


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,purchase_year,purchase_month,...,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,items_quantity,total_price,total_freight,payment_total,payment_methods_count,payment_types
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,Delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2017,10,...,7c396fd4830fd04220f754e42b4e5bff,3149,Sao Paulo,SP,1.0,29.99,8.72,38.71,2.0,"credit_card, voucher"
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,Delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,2018,7,...,af07308b275d755c9edb36a90c618231,47813,Barreiras,BA,1.0,118.70,22.76,141.46,1.0,boleto
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,Delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,2018,8,...,3a653a41f6f9fc3d2a113cf8398680e8,75265,Vianopolis,GO,1.0,159.90,19.22,179.12,1.0,credit_card
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,Delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,2017,11,...,7c142cf63193a1473d2e66489a9ae977,59296,Sao Goncalo Do Amarante,RN,1.0,45.00,27.20,72.20,1.0,credit_card
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,Delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2018,2,...,72632f0f9dd73dfee390c9b22eb56dd6,9195,Santo Andre,SP,1.0,19.90,8.72,28.62,1.0,credit_card
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,Delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28,2017,3,...,6359f309b166b0196dbf7ad2ac62bb5a,12209,Sao Jose Dos Campos,SP,1.0,72.00,13.08,85.08,1.0,credit_card
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,Delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02,2018,2,...,da62f9e57a76d978d02ab5362c509660,11722,Praia Grande,SP,1.0,174.90,20.10,195.00,1.0,credit_card
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,Delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27,2017,8,...,737520a9aad80b3fbbdad19b66b37b30,45920,Nova Vicosa,BA,1.0,205.99,65.02,271.01,1.0,credit_card
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,Delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15,2018,1,...,5097a5312c8b157bb7be58ae360ef43c,28685,Japuiba,RJ,2.0,359.98,81.18,441.16,1.0,credit_card


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,purchase_year,purchase_month,...,items_quantity,total_price,total_freight,payment_total,payment_methods_count,payment_types,review_score_mean,review_score_max,review_score_min,review_count
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,Delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2017,10,...,1.0,29.99,8.72,38.71,2.0,"credit_card, voucher",4.0,4.0,4.0,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,Delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,2018,7,...,1.0,118.70,22.76,141.46,1.0,boleto,4.0,4.0,4.0,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,Delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,2018,8,...,1.0,159.90,19.22,179.12,1.0,credit_card,5.0,5.0,5.0,1.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,Delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,2017,11,...,1.0,45.00,27.20,72.20,1.0,credit_card,5.0,5.0,5.0,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,Delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2018,2,...,1.0,19.90,8.72,28.62,1.0,credit_card,5.0,5.0,5.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,Delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28,2017,3,...,1.0,72.00,13.08,85.08,1.0,credit_card,5.0,5.0,5.0,1.0
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,Delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02,2018,2,...,1.0,174.90,20.10,195.00,1.0,credit_card,4.0,4.0,4.0,1.0
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,Delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27,2017,8,...,1.0,205.99,65.02,271.01,1.0,credit_card,5.0,5.0,5.0,1.0
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,Delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15,2018,1,...,2.0,359.98,81.18,441.16,1.0,credit_card,2.0,2.0,2.0,1.0


In [21]:
# Bloco 6: exportação dos datasets prontos para o Tableau
# Os CSVs aqui são gerados para atender às perguntas de negócio do arquivo de entendimento.

# 1. Base de pedido: 1 linha por pedido, ideal para visão macro e KPIs
tableau_order_base = orders_items_payments_reviews.copy()
tableau_order_base["customer_region"] = tableau_order_base["customer_state"].astype(str)
tableau_order_base["delivery_status"] = np.where(tableau_order_base["delay_days"] > 0, "Atrasado", "NoPrazo")
tableau_order_base["avg_review_score"] = tableau_order_base["review_score_mean"].fillna(0)
tableau_order_base["revenue_total"] = tableau_order_base["payment_total"].fillna(0)
tableau_order_base["ticket_medio"] = np.where(
    tableau_order_base["items_quantity"].fillna(0) > 0,
    tableau_order_base["payment_total"] / tableau_order_base["items_quantity"],
    0
)
tableau_order_base["order_month_name"] = tableau_order_base["purchase_month_name"]
tableau_order_base["order_weekday"] = tableau_order_base["purchase_weekday_name"]
tableau_order_base["order_year"] = tableau_order_base["purchase_year"]
tableau_order_base = tableau_order_base.sort_values("order_purchase_timestamp")

# 2. Base detalhada por item: 1 linha por item
tableau_item_base = model_df.copy()
tableau_item_base["customer_region"] = tableau_item_base["customer_state"].astype(str)
tableau_item_base["seller_region"] = tableau_item_base["seller_state"].astype(str)
tableau_item_base["delivery_status"] = np.where(tableau_item_base["delay_days"] > 0, "Atrasado", "NoPrazo")
tableau_item_base["item_revenue"] = tableau_item_base["price"] * tableau_item_base["quantity"]

# Exportar bases analíticas
model_df.to_csv(output_path / "orders_customer_product_seller_full.csv",
    index=False, 
    sep=";",
    decimal=",",
    encoding="utf-8-sig")
orders_items_payments_reviews.to_csv(output_path / "orders_customer_financial_analytics.csv",
    index=False, 
    sep=";",
    decimal=",",
    encoding="utf-8-sig")
df_orders.to_csv(output_path / "orders_temporal_features.csv",
    index=False, 
    sep=";",
    decimal=",",
    encoding="utf-8-sig")

print(model_df.info())
print(orders_items_payments_reviews.info())
print(df_orders.info())

<class 'pandas.DataFrame'>
RangeIndex: 113425 entries, 0 to 113424
Data columns (total 62 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       113425 non-null  str           
 1   customer_id                    113425 non-null  str           
 2   order_status                   113425 non-null  str           
 3   order_purchase_timestamp       113425 non-null  datetime64[us]
 4   order_approved_at              113264 non-null  datetime64[us]
 5   order_delivered_carrier_date   111457 non-null  datetime64[us]
 6   order_delivered_customer_date  110196 non-null  datetime64[us]
 7   order_estimated_delivery_date  113425 non-null  datetime64[us]
 8   purchase_year                  113425 non-null  int32         
 9   purchase_month                 113425 non-null  int32         
 10  purchase_month_name            113425 non-null  str           
 11  purchase_da

In [22]:
# Bloco de diagnóstico: verificar tipos de dados e nulos inesperados
# Este bloco ajuda a identificar problemas antes da exportação final.

print("=" * 80)
print("DIAGNÓSTICO - tableau_order_base")
print("=" * 80)
print(tableau_order_base.dtypes)
print("\nNulos por coluna:")
print(tableau_order_base.isnull().sum()[tableau_order_base.isnull().sum() > 0])

print("\n" + "=" * 80)
print("DIAGNÓSTICO - tableau_item_base")
print("=" * 80)
print(tableau_item_base.dtypes)
print("\nNulos por coluna:")
print(tableau_item_base.isnull().sum()[tableau_item_base.isnull().sum() > 0])

DIAGNÓSTICO - tableau_order_base
order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
purchase_year                             int32
purchase_month                            int32
purchase_month_name                         str
purchase_day                              int32
purchase_dayofweek                        int32
purchase_weekday_name                       str
purchase_hour                             int32
approval_year                           float64
approval_month                          float64
approval_day                            float64
carrier_year                            float64
carrier_month                           float64
carrier

In [23]:
# Bloco de validação e limpeza de tipos para consumo no Tableau
# Este bloco garante que:
# - zip_codes sejam strings com zeros à esquerda
# - colunas inteiras sejam reconhecidas como inteiras
# - nulos esperados sejam documentados

# Função para restaurar zip_code com zeros à esquerda
def format_zipcode(value):
    """Converte número ou string para formato de CEP com 5 dígitos."""
    if pd.isna(value):
        return None
    return str(int(float(value))).zfill(5)

# Processamento de tableau_order_base
print("Limpando tableau_order_base...")
tableau_order_base_clean = tableau_order_base.copy()

# Converter zip_code de volta para string com zeros à esquerda
if 'customer_zip_code_prefix' in tableau_order_base_clean.columns:
    tableau_order_base_clean['customer_zip_code_prefix'] = (
        tableau_order_base_clean['customer_zip_code_prefix']
        .apply(format_zipcode)
        .astype('string')
    )

# Processamento de tableau_item_base
print("Limpando tableau_item_base...")
tableau_item_base_clean = tableau_item_base.copy()

# Converter zip_codes para string com zeros à esquerda
zipcode_cols = ['customer_zip_code_prefix', 'seller_zip_code_prefix']
for col in zipcode_cols:
    if col in tableau_item_base_clean.columns:
        tableau_item_base_clean[col] = (
            tableau_item_base_clean[col]
            .apply(format_zipcode)
            .astype('string')
        )
        
# Exportar bases limpas em CSV com tipos corretos
print("\nExportando bases limpas para o Tableau...")
tableau_order_base_clean.to_csv(output_path / "tableau_order_base.csv",
    index=False, 
    sep=";",
    decimal=",",
    encoding="utf-8-sig")
tableau_item_base_clean.to_csv(output_path / "tableau_item_base.csv",
    index=False, 
    sep=";",
    decimal=",",
    encoding="utf-8-sig")

print("✓ Exportação concluída com sucesso!")
print("\nResumo dos tipos após limpeza - tableau_order_base:")
print(tableau_order_base_clean.dtypes)
print("\nNulos após limpeza - tableau_order_base:")
print(tableau_order_base_clean.isnull().sum()[tableau_order_base_clean.isnull().sum() > 0])

Limpando tableau_order_base...
Limpando tableau_item_base...

Exportando bases limpas para o Tableau...
✓ Exportação concluída com sucesso!

Resumo dos tipos após limpeza - tableau_order_base:
order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
purchase_year                             int32
purchase_month                            int32
purchase_month_name                         str
purchase_day                              int32
purchase_dayofweek                        int32
purchase_weekday_name                       str
purchase_hour                             int32
approval_year                           float64
approval_month                         

In [24]:
# Bloco de validação final: verificar que zip_codes foram formatados corretamente
print("="*80)
print("VALIDAÇÃO FINAL - ZIP CODES")
print("="*80)
print("\nExemplos de customer_zip_code_prefix (tableau_order_base):")
print(tableau_order_base_clean['customer_zip_code_prefix'].dropna().head(10).values)
print("\nExemplos de seller_zip_code_prefix (tableau_item_base):")
print(tableau_item_base_clean['seller_zip_code_prefix'].dropna().head(10).values)

print("\n" + "="*80)
print("VALIDAÇÃO FINAL - TIPOS DE DADOS")
print("="*80)
print("\nColunas com tipo Int64 (inteiro nullable) em tableau_order_base:")
int64_cols = [col for col, dtype in tableau_order_base_clean.dtypes.items() if dtype == 'Int64']
print(int64_cols)

print("\nColunas com tipo Int64 (inteiro nullable) em tableau_item_base:")
int64_cols_item = [col for col, dtype in tableau_item_base_clean.dtypes.items() if dtype == 'Int64']
print(int64_cols_item)

print("\n" + "="*80)
print("✓ BASES PRONTAS PARA TABLEAU")
print("="*80)
print(f"Arquivo: {output_path / 'tableau_order_base.csv'}")
print(f"Tamanho: {tableau_order_base_clean.shape[0]} linhas, {tableau_order_base_clean.shape[1]} colunas")
print(f"\nArquivo: {output_path / 'tableau_item_base.csv'}")
print(f"Tamanho: {tableau_item_base_clean.shape[0]} linhas, {tableau_item_base_clean.shape[1]} colunas")

VALIDAÇÃO FINAL - ZIP CODES

Exemplos de customer_zip_code_prefix (tableau_order_base):
<StringArray>
['69309', '99025', '12244', '14600', '02975', '04106', '98280', '22770',
 '90040', '13185']
Length: 10, dtype: string

Exemplos de seller_zip_code_prefix (tableau_item_base):
<StringArray>
['09350', '31570', '14840', '31842', '08752', '07112', '05455', '12940',
 '13720', '08577']
Length: 10, dtype: string

VALIDAÇÃO FINAL - TIPOS DE DADOS

Colunas com tipo Int64 (inteiro nullable) em tableau_order_base:
[]

Colunas com tipo Int64 (inteiro nullable) em tableau_item_base:
[]

✓ BASES PRONTAS PARA TABLEAU
Arquivo: C:\Users\Thiago Laizy e Bofe\Documents\GitHub\E-commerce-analytics\data\output\tableau_order_base.csv
Tamanho: 99441 linhas, 54 colunas

Arquivo: C:\Users\Thiago Laizy e Bofe\Documents\GitHub\E-commerce-analytics\data\output\tableau_item_base.csv
Tamanho: 113425 linhas, 65 colunas


In [25]:
# Bloco de documentação: justificar nulos no modelo de negócio
# Este bloco explica quais nulos são ESPERADOS e fazem sentido no contexto de negócio

nulos_esperados = """
JUSTIFICATIVA DOS NULOS ESPERADOS NO MODELO DE NEGÓCIO:

1. **order_approved_at e derivadas (approval_year, approval_month, approval_day)**
   - ~160 registros nulos
   - Motivo: Pedidos cancelados ou com erro de processamento não têm data de aprovação
   - Impacto: Filtros no Tableau devem ignorar nulos ou criar categoria \"Sem aprovação\"

2. **order_delivered_carrier_date e derivadas**
   - ~1783 registros nulos
   - Motivo: Pedidos cancelados ou ainda não enviados não têm data de coleta pela transportadora
   - Impacto: Esperado em pedidos com status \"Canceled\" ou \"Processing\"

3. **order_delivered_customer_date, delivery_days, delay_days e derivadas**
   - ~2965-3229 registros nulos
   - Motivo: Pedidos não entregues (cancelados, devolvidos ou em trânsito) não têm data de entrega
   - Impacto: KPIs de entrega devem filtrar estes registros ou calcular médias excluindo nulos

4. **items_quantity, total_price, total_freight**
   - ~775 registros nulos
   - Motivo: Estes registros provavelmente correspondem a pedidos sem itens (erro de dados ou cancelamento)
   - Impacto: Devem ser investigados ou excluídos de análises de receita

5. **review_score_mean, review_score_max, review_score_min, review_count**
   - ~768-961 registros nulos
   - Motivo: Nem todos os pedidos recebem avaliação de cliente
   - Impacto: Análises de satisfação devem filtrar registros com avaliação disponível

6. **payment_total, payment_methods_count, payment_types**
   - ~1 registro nulo
   - Motivo: Exceção, provavelmente dado corrompido ou cancelado antes do pagamento
   - Impacto: Investigue manualmente ou trate como caso especial

RECOMENDAÇÕES PARA DASHBOARD NO TABLEAU:
- Use filtros com \"Include null values\" apenas para análises exploratórias
- Para KPIs críticos, crie uma métrica calculada que exclua nulos: SUM(revenue) / COUNT(IF NOT NULL(revenue))
- Crie dimensões booleanas para status de entrega, pagamento e avaliação para facilitar drilldown
"""

print(nulos_esperados)

# Salvar documentação em arquivo
doc_file = project_root / "docs" / "data_cleaning_notes.md"
with open(doc_file, 'w', encoding='utf-8') as f:
    f.write(nulos_esperados)
    
print(f"\n✓ Documentação salva em: {doc_file}")


JUSTIFICATIVA DOS NULOS ESPERADOS NO MODELO DE NEGÓCIO:

1. **order_approved_at e derivadas (approval_year, approval_month, approval_day)**
   - ~160 registros nulos
   - Motivo: Pedidos cancelados ou com erro de processamento não têm data de aprovação
   - Impacto: Filtros no Tableau devem ignorar nulos ou criar categoria "Sem aprovação"

2. **order_delivered_carrier_date e derivadas**
   - ~1783 registros nulos
   - Motivo: Pedidos cancelados ou ainda não enviados não têm data de coleta pela transportadora
   - Impacto: Esperado em pedidos com status "Canceled" ou "Processing"

3. **order_delivered_customer_date, delivery_days, delay_days e derivadas**
   - ~2965-3229 registros nulos
   - Motivo: Pedidos não entregues (cancelados, devolvidos ou em trânsito) não têm data de entrega
   - Impacto: KPIs de entrega devem filtrar estes registros ou calcular médias excluindo nulos

4. **items_quantity, total_price, total_freight**
   - ~775 registros nulos
   - Motivo: Estes registros prov